In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from pathlib import Path

WORKSPACE = Path("/content/drive/MyDrive/image-captioning-rag")
RAW_ROOT = WORKSPACE / "00_original_data"

assert WORKSPACE.exists(), "Workspace not found"
assert RAW_ROOT.exists(), "Original dataset directory not found"

TRAIN_IMAGES = RAW_ROOT / "train2017"
VAL_IMAGES = RAW_ROOT / "val2017"
ANNOTATIONS = RAW_ROOT / "annotations"

assert TRAIN_IMAGES.exists(), f"Training images not found: {TRAIN_IMAGES}"
assert VAL_IMAGES.exists(), f"Validation images not found: {VAL_IMAGES}"
assert ANNOTATIONS.exists(), f"Annotations directory not found: {ANNOTATIONS}"

print("Workspace and dataset paths verified.")
print("Training images:", TRAIN_IMAGES)
print("Validation images:", VAL_IMAGES)
print("Annotations:", ANNOTATIONS)

Workspace and dataset paths verified.
Training images: /content/drive/MyDrive/image-captioning-rag/00_original_data/train2017
Validation images: /content/drive/MyDrive/image-captioning-rag/00_original_data/val2017
Annotations: /content/drive/MyDrive/image-captioning-rag/00_original_data/annotations


In [4]:
import antigravity

In [5]:
import json
from pathlib import Path

# Adjust this path if your COCO annotations are stored in a different subfolder
ANNOTATIONS = Path("/content/drive/MyDrive/COCO/annotations")

# Fallback/Check: If the directory does not exist, search Drive for it
if not ANNOTATIONS.exists():
    print("Searching Google Drive for 'annotations' folder...")
    drive_root = Path("/content/drive/MyDrive")
    possible_paths = list(drive_root.glob("**/annotations"))
    if possible_paths:
        ANNOTATIONS = possible_paths[0]
        print(f"Found annotations at: {ANNOTATIONS}")
    else:
        print("Could not find an 'annotations' folder in your Drive automatically.")
        print("Please set ANNOTATIONS = Path('/path/to/your/annotations') manually.")

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

if ANNOTATIONS.exists():
    CAPTIONS_TRAIN = load_json(
        ANNOTATIONS / "captions_train2017.json"
    )
    CAPTIONS_VAL = load_json(
        ANNOTATIONS / "captions_val2017.json"
    )
    INSTANCES_TRAIN = load_json(
        ANNOTATIONS / "instances_train2017.json"
    )
    INSTANCES_VAL = load_json(
        ANNOTATIONS / "instances_val2017.json"
    )

    print("Train caption annotations:",
          len(CAPTIONS_TRAIN["annotations"]))
    print("Validation caption annotations:",
          len(CAPTIONS_VAL["annotations"]))
    print("Train object annotations:",
          len(INSTANCES_TRAIN["annotations"]))
    print("Validation object annotations:",
          len(INSTANCES_VAL["annotations"]))


Searching Google Drive for 'annotations' folder...
Found annotations at: /content/drive/MyDrive/image-captioning-rag/00_original_data/annotations
Train caption annotations: 591753
Validation caption annotations: 25014
Train object annotations: 860001
Validation object annotations: 36781


In [6]:
print("Image record:", CAPTIONS_TRAIN["images"][0])
print("Caption record:", CAPTIONS_TRAIN["annotations"][0])
print("Object record:", INSTANCES_TRAIN["annotations"][0])

Image record: {'license': 3, 'file_name': '000000391895.jpg', 'coco_url': 'http://images.cocodataset.org/train2017/000000391895.jpg', 'height': 360, 'width': 640, 'date_captured': '2013-11-14 11:18:45', 'flickr_url': 'http://farm9.staticflickr.com/8186/8119368305_4e622c8349_z.jpg', 'id': 391895}
Caption record: {'image_id': 203564, 'id': 37, 'caption': 'A bicycle replica with a clock as the front wheel.'}
Object record: {'segmentation': [[239.97, 260.24, 222.04, 270.49, 199.84, 253.41, 213.5, 227.79, 259.62, 200.46, 274.13, 202.17, 277.55, 210.71, 249.37, 253.41, 237.41, 264.51, 242.54, 261.95, 228.87, 271.34]], 'area': 2765.1486500000005, 'iscrowd': 0, 'image_id': 558840, 'bbox': [199.84, 200.46, 77.71, 70.88], 'category_id': 58, 'id': 156}


In [7]:
from collections import Counter

def caption_counts(caption_data):
    return Counter(
        ann["image_id"]
        for ann in caption_data["annotations"]
    )

train_caption_counts = caption_counts(CAPTIONS_TRAIN)
val_caption_counts = caption_counts(CAPTIONS_VAL)

train_ids = {img["id"] for img in CAPTIONS_TRAIN["images"]}
val_ids = {img["id"] for img in CAPTIONS_VAL["images"]}

print("Unique train images:", len(train_ids))
print("Unique val images:", len(val_ids))
print("Train images with five captions:",
      sum(train_caption_counts[i] == 5 for i in train_ids))
print("Val images with five captions:",
      sum(val_caption_counts[i] == 5 for i in val_ids))

Unique train images: 118287
Unique val images: 5000
Train images with five captions: 117972
Val images with five captions: 4987


In [8]:
from PIL import Image, ImageOps
from tqdm.auto import tqdm

def validate_image_paths(image_dir, image_records):
    missing = []
    corrupt = []

    for record in tqdm(image_records):
        path = image_dir / record["file_name"]

        if not path.exists():
            missing.append(record["id"])
            continue

        try:
            with Image.open(path) as im:
                im.verify()
        except Exception:
            corrupt.append(record["id"])

    return {
        "missing_ids": missing,
        "corrupt_ids": corrupt,
    }